# 05 · Features (Ludovico)

- **Part A · Feature engineering:** which new features, built from what the EDA found, help the models?
- **Part B · Feature selection:** which columns can be removed, and does removing them help?

**How it is organised**, so that configurations can be changed and re-run quickly:
- **Every feature group is a registered function** `df -> df`. To try an idea you write one function and give it a name.
- **`FeatureBuilder`** is the first step of the pipeline. Its parameters say which groups to add and which columns to drop.
- **Experiments are data:** a dictionary `{name: configuration}`. `compare()` scores all of them on the same folds, returns the paired table and saves the results to an experiment log.

**Method** (as in `preprocessing_ludovico` and the week 4 practical): the same 8 half-year time folds, two measuring instruments (logistic regression with the linear recipe, gradient boosting with the tree recipe), paired differences against the reference, SEM and folds won. A feature group is kept only if it helps clearly.

**Leakage rule:** every feature uses **only its own row** (and the static customer table). Nothing is learned from other rows, so applying it before the split leaks nothing.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. The framework</font>](#framework)
* [<font color='#E8800A'>2. The feature groups</font>](#groups)
* [<font color='#E8800A'>3. Each group on its own</font>](#alone)
* [<font color='#E8800A'>4. Combining the useful groups</font>](#combine)
* [<font color='#E8800A'>5. Effect on the official validation</font>](#official)
* [<font color='#E8800A'>6. Summary of part A</font>](#summary)
* [<font color='#E8800A'>7. Part B: the reference and the selectors</font>](#selection)
* [<font color='#E8800A'>8. Filter methods</font>](#filter)
* [<font color='#E8800A'>9. Wrapper method</font>](#wrapper)
* [<font color='#E8800A'>10. Embedded methods</font>](#embedded)
* [<font color='#E8800A'>11. The random_noise floor</font>](#noise)
* [<font color='#E8800A'>12. Vote and final board</font>](#vote)
* [<font color='#E8800A'>13. Summary of part B</font>](#summaryb)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

`%run` executes `preprocessing_ludovico.ipynb` (and, through it, validation and cleaning). This gives back the clean data, the 8 preprocessing folds (`prep_folds`), the official folds, the two recipes and the paired-table helpers. It takes about 2 minutes.

In [1]:
%%capture
%run ./preprocessing_ludovico.ipynb

In [2]:
import json
from datetime import datetime

from sklearn.model_selection import cross_validate

from src.paths import DATA_PROCESSED

# Every column except the target: the identifiers and the date are needed to BUILD some features,
# and FeatureBuilder drops them before the model sees the data
X_full = train.drop(columns=[TARGET])
X_test_full = test.copy()
ALWAYS_DROP = ["customer_id", "purchase_date", "random_noise"]

# <font color='#E8800A'>1. The framework</font> <a class="anchor" id="framework"></a>
[Back to TOC](#toc)

**Four pieces:**
1. `FEATURE_GROUPS`: the registry. `@feature_group("name")` adds a function to it.
2. `FeatureBuilder`: adds the chosen groups, then drops identifiers and any columns listed in `drop`.
3. `feature_pipeline`: `FeatureBuilder` followed by the recipe of the model family. New columns are routed automatically by type (numeric or categorical).
4. `compare`: scores a dictionary of configurations on the same folds and returns the paired table.

**One change to the linear recipe:** some new features can be negative (e.g. a margin), and `log1p` is undefined below −1. The numeric transform becomes a **signed log**, `sign(x)·log(1+|x|)`. It is identical to `log1p` on the existing columns, which are all ≥ 0, so the results of `preprocessing_ludovico` do not change.

In [3]:
FEATURE_GROUPS = {}


def feature_group(name):
    """Register a function df -> df that adds columns computed from each row (no learning)."""
    def register(func):
        FEATURE_GROUPS[name] = func
        return func
    return register


def safe_div(numerator, denominator):
    """Division that gives NaN (later imputed) instead of inf when the denominator is 0."""
    return (numerator / denominator.where(denominator != 0)).astype(float)


class FeatureBuilder(BaseEstimator, TransformerMixin):
    """Pipeline step: add the chosen feature groups, then drop identifiers and the listed columns."""

    def __init__(self, groups=(), drop=(), keep=()):
        self.groups = groups
        self.drop = drop
        self.keep = keep  # columns of ALWAYS_DROP to let through (used for the random_noise floor)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        for name in self.groups:
            X = FEATURE_GROUPS[name](X)
        always = [c for c in ALWAYS_DROP if c not in self.keep]
        return X.drop(columns=[c for c in [*always, *self.drop] if c in X.columns])


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


RECIPES = {
    "logistic regression": lambda: linear_recipe(LogisticRegression(max_iter=2000)).set_params(
        prep__num__transform=FunctionTransformer(signed_log1p)),
    "gradient boosting": lambda: tree_recipe(HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
}


def feature_pipeline(config, model_name):
    """FeatureBuilder(config) followed by the preprocessing recipe and the model of `model_name`."""
    return Pipeline([("features", FeatureBuilder(**config)), *RECIPES[model_name]().steps])

In [4]:
EXPERIMENT_LOG = DATA_PROCESSED / "feature_experiments.csv"


def compare(configs, model_name, reference="base", folds=None):
    """Score every configuration on the same folds (ROC AUC), log the results, return the paired table."""
    folds = prep_folds if folds is None else folds
    scores = {}
    for name, config in configs.items():
        cv = cross_validate(feature_pipeline(config, model_name), X_full, y, cv=folds,
                            scoring="roc_auc", n_jobs=-1)
        scores[name] = cv["test_score"]
    scores = pd.DataFrame(scores, index=pd.RangeIndex(1, len(folds) + 1, name="fold"))

    stamp = datetime.now().isoformat(timespec="seconds")
    log = pd.DataFrame([{"time": stamp, "model": model_name, "experiment": name,
                         "config": json.dumps(configs[name]), "folds": len(folds),
                         "roc_auc_mean": scores[name].mean(), "roc_auc_folds": json.dumps(scores[name].round(5).tolist())}
                        for name in scores])
    log.to_csv(EXPERIMENT_LOG, mode="a", header=not EXPERIMENT_LOG.exists(), index=False)
    return paired_table(scores, reference)

**Sanity check:** with no feature group, the pipeline must reproduce the results of `preprocessing_ludovico` on the same folds. Logistic regression is now on the signed log, which is identical on non-negative data.

In [5]:
check = {name: cross_validate(feature_pipeline({}, name), X_full, y, cv=official_folds,
                              scoring="roc_auc")["test_score"].mean() for name in RECIPES}
expected = {"logistic regression": recipes.loc[("logistic regression", "recipe"), "ROC AUC mean"],
            "gradient boosting": recipes.loc[("gradient boosting", "recipe"), "ROC AUC mean"]}
for name in RECIPES:
    assert abs(check[name] - expected[name]) < 1e-9, (name, check[name], expected[name])
print("framework reproduces preprocessing_ludovico:", {k: round(v, 4) for k, v in check.items()})

framework reproduces preprocessing_ludovico: {'logistic regression': np.float64(0.7829), 'gradient boosting': np.float64(0.7625)}


# <font color='#E8800A'>2. The feature groups</font> <a class="anchor" id="groups"></a>
[Back to TOC](#toc)

**Every group comes from an EDA finding or a business hypothesis.** All are computed from the row itself.

| Group | New columns | Hypothesis |
|---|---|---|
| `recency_ratio` | days since last purchase ÷ usual gap | a customer **late** compared with their own rhythm is less likely to come back (EDA: recency is the strongest signal) |
| `activity_trend` | share of the last-365-day purchases and spend that fall in the last 90 days | a customer who is **speeding up** comes back, one who is slowing down does not |
| `intensity` | purchases per year of tenure; current purchase ÷ usual purchase value | frequency normalised by how long the customer has existed; an unusually large order may cover future needs |
| `invoice` | margin, discount share, zero-tax flag, shipping flag | invoice profile (export / exempt customers, discounted orders) (EDA Step 6) |
| `calendar` | month of the purchase (categorical) | seasonality. Dates are anonymised, but order and intervals are preserved |
| `cycle_position` | position of the month in a 4-month cycle (month mod 4, categorical) | added after a first run: the repeat rate follows a 4-month cycle (see section 3), so 4 well-populated levels may be enough instead of 12 |
| `customer_since` | years the customer existed before their first recorded purchase (`clientes.csv`) | older customers repeat more (EDA Step 23), beyond what `tenure` already says |
| `customer_geo` | country and commercial region (`clientes.csv`) | regional differences (EDA Step 23: two countries ≈ 0.56 vs ≈ 0.39) |

**`clientes.csv` check before using it.** It is a snapshot of today, so an attribute could contain information from after the purchase. `customer_since_year` must never be later than the purchase, or than the customer's first recorded purchase.

In [6]:
CLIENTES = pd.read_csv(DATA_RAW / "clientes.csv").set_index("customer_id")[
    ["customer_since_year", "country_or_region", "commercial_region"]]

since = X_full["customer_id"].map(CLIENTES["customer_since_year"])
first_purchase_year = (X_full["purchase_date"] - pd.to_timedelta(X_full["customer_tenure_days"], unit="D")).dt.year
pd.Series({
    "train rows matched in clientes.csv": X_full["customer_id"].isin(CLIENTES.index).mean(),
    "since_year later than the purchase year": (since > X_full["purchase_date"].dt.year).mean(),
    "since_year later than the first recorded purchase": (since > first_purchase_year).mean(),
    "since_year earlier than the first recorded purchase": (since < first_purchase_year).mean(),
}, name="share of rows").to_frame().style.format("{:.1%}")

,share of rows
train rows matched in clientes.csv,100.0%
since_year later than the purchase year,0.0%
since_year later than the first recorded purchase,0.0%
since_year earlier than the first recorded purchase,31.2%


**No sign of future information:** `customer_since_year` is never later than the purchase or than the first recorded purchase. In about a third of the rows it is **earlier**: the customer existed before their first purchase in this data. That is information `customer_tenure_days` does not have, and `customer_since` encodes exactly that gap.

In [7]:
@feature_group("recency_ratio")
def recency_ratio(df):
    return df.assign(recency_ratio=safe_div(df["days_since_previous_purchase"], df["prior_mean_gap_days"]))


@feature_group("activity_trend")
def activity_trend(df):
    return df.assign(
        count_share_90_of_365=safe_div(df["purchase_count_90d"], df["purchase_count_365d"]),
        spend_share_90_of_365=safe_div(df["spend_90d"], df["spend_365d"]),
    )


@feature_group("intensity")
def intensity(df):
    return df.assign(
        purchases_per_year=safe_div(df["prior_purchase_count"], df["customer_tenure_days"] / 365.25),
        current_vs_usual_value=safe_div(df["total_net"], df["prior_mean_purchase_value"]),
    )


@feature_group("invoice")
def invoice(df):
    return df.assign(
        margin=1 - safe_div(df["cost"], df["total_net"]),
        discount_share=safe_div(df["discount_line"] + df["discount_global"], df["total_gross"]),
        zero_tax=(df["total_tax"] == 0).astype(int),
        has_shipping=(df["total_shipping"] > 0).astype(int),
    )


@feature_group("calendar")
def calendar(df):
    return df.assign(purchase_month=df["purchase_date"].dt.month.astype(str))  # categorical: one-hot


@feature_group("cycle_position")
def cycle_position(df):
    # months 3, 7, 11 -> "3"; 1, 5, 9 -> "1"; ... : one level per position in the 4-month cycle
    return df.assign(cycle_position=(df["purchase_date"].dt.month % 4).astype(str))


@feature_group("customer_since")
def customer_since(df):
    first_year = (df["purchase_date"] - pd.to_timedelta(df["customer_tenure_days"], unit="D")).dt.year
    return df.assign(years_before_first_purchase=first_year - df["customer_id"].map(CLIENTES["customer_since_year"]))


@feature_group("customer_geo")
def customer_geo(df):
    return df.assign(country=df["customer_id"].map(CLIENTES["country_or_region"]).astype(str),
                     commercial_region=df["customer_id"].map(CLIENTES["commercial_region"]).astype(str))


# Every group must run on train and test without errors and add only new columns
for name, func in FEATURE_GROUPS.items():
    added = func(X_full.head(50)).columns.difference(X_full.columns)
    func(X_test_full.head(50))
    print(f"{name:16s} -> {', '.join(added)}")

recency_ratio    -> recency_ratio
activity_trend   -> count_share_90_of_365, spend_share_90_of_365
intensity        -> current_vs_usual_value, purchases_per_year
invoice          -> discount_share, has_shipping, margin, zero_tax
calendar         -> purchase_month
cycle_position   -> cycle_position
customer_since   -> years_before_first_purchase
customer_geo     -> commercial_region, country


# <font color='#E8800A'>3. Each group on its own</font> <a class="anchor" id="alone"></a>
[Back to TOC](#toc)

**First question:** does each group, **added on its own** to the current features, help? Each configuration below is the base plus one group. The reference is `base`, the features used so far.

In [8]:
alone = {"base": {}} | {f"+ {name}": {"groups": [name]} for name in FEATURE_GROUPS}

alone_tables = {model_name: compare(alone, model_name) for model_name in RECIPES}
show(alone_tables["logistic regression"])

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
+ cycle_position,0.7973,+0.0064,0.0028,+2.3,6/8
+ calendar,0.7970,+0.0060,0.0038,+1.6,6/8
+ customer_since,0.7916,+0.0006,0.0006,+1.1,6/8
+ recency_ratio,0.7911,+0.0001,0.0002,+0.6,4/8
base,0.7910,+0.0000,-,-,ref
+ invoice,0.7909,-0.0000,0.0010,-0.0,5/8
+ intensity,0.7908,-0.0001,0.0003,-0.4,3/8
+ activity_trend,0.7908,-0.0001,0.0002,-0.6,4/8
+ customer_geo,0.7903,-0.0007,0.0013,-0.5,5/8


In [9]:
show(alone_tables["gradient boosting"])

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
+ calendar,0.7859,+0.0131,0.0062,+2.1,6/8
+ cycle_position,0.7786,+0.0058,0.0035,+1.7,5/8
+ recency_ratio,0.7774,+0.0046,0.0035,+1.3,6/8
+ intensity,0.7762,+0.0034,0.0038,+0.9,5/8
+ activity_trend,0.7748,+0.0020,0.0017,+1.2,5/8
+ customer_geo,0.7735,+0.0007,0.0030,+0.3,3/8
base,0.7728,+0.0000,-,-,ref
+ customer_since,0.7715,-0.0013,0.0030,-0.4,4/8
+ invoice,0.7710,-0.0018,0.0038,-0.5,3/8


**Why the month matters:** the repeat rate by month of purchase, in two halves of the train period.

In [10]:
by_month = X_full.assign(month=X_full["purchase_date"].dt.month, target=y,
                         period=np.where(X_full["purchase_date"].dt.year <= 2019, "2015-19", "2020-25"))
monthly = by_month.pivot_table(index="period", columns="month", values="target", aggfunc="mean")
monthly.loc["rows (all years)"] = by_month["month"].value_counts().sort_index()
monthly.style.format("{:.2f}").format("{:.0f}", subset=pd.IndexSlice[["rows (all years)"], :])

month,1,2,3,4,5,6,7,8,9,10,11,12
period,,,,,,,,,,,,
2015-19,0.68,0.53,0.37,0.47,0.75,0.70,0.52,0.61,0.72,0.63,0.45,0.59
2020-25,0.69,0.65,0.48,0.54,0.72,0.57,0.46,0.61,0.67,0.64,0.48,0.59
rows (all years),444,260,876,442,196,311,874,525,436,680,884,541


**Reading the boards**
- **Only the month of purchase helps clearly, and it helps both models:** `calendar` gives +0.0060 to logistic regression (6/8 folds) and **+0.0131** to gradient boosting (2.1 SEM, 6/8).
- **The reason is a 4-month cycle**, the same in 2015–19 and 2020–25 (table above). March, July and November have the **most purchases** and the **lowest repeat rate** (~0.45). January, May and September, two months before each peak, have the highest (~0.70). This fits large orders placed every 4 months: a purchase in a peak month is followed by the next peak more than 90 days later (target 0), while a purchase two months before a peak is followed by it within 90 days (target 1).
- **`cycle_position` confirms the mechanism:** 4 levels (month mod 4) give logistic regression as much as the 12 months (+0.0064, 2.3 SEM). Gradient boosting gains less from it (+0.0058) than from the full month: trees can use the extra detail.
- **Every other group gives logistic regression nothing** (all within ±0.001). For gradient boosting, `recency_ratio`, `intensity` and `activity_trend` are slightly positive (+0.002 to +0.005, 0.9–1.3 SEM). That is not enough on their own, so they are re-tested next to the month in section 4.
- **`invoice`, `customer_since` and `customer_geo` do not help** either model. The hypotheses from the EDA (margin, older customers, country) do not add anything beyond what the existing columns already say (`customer_tenure_days` already captures seniority).

# <font color='#E8800A'>4. Combining the useful groups</font> <a class="anchor" id="combine"></a>
[Back to TOC](#toc)

**Second question:** once the month is in, do the other groups still add something? Section 3 tested each group **on its own**. A group can look useful alone and add nothing next to another one that carries the same information. Configurations:
- **selected:** the groups kept in section 3;
- **selected + recency / + boosting candidates:** the groups that were slightly positive for gradient boosting in section 3 (`recency_ratio`, `intensity`, `activity_trend`), added to the selection;
- **all groups:** every group together.

In [11]:
SELECTED = ['calendar']

BOOSTING_CANDIDATES = ["recency_ratio", "intensity", "activity_trend"]
combine = {
    "base": {},
    "selected": {"groups": SELECTED},
    "selected + recency_ratio": {"groups": SELECTED + ["recency_ratio"]},
    "selected + boosting candidates": {"groups": SELECTED + BOOSTING_CANDIDATES},
    "all groups": {"groups": list(FEATURE_GROUPS)},
}

combine_tables = {model_name: compare(combine, model_name) for model_name in RECIPES}
show(combine_tables["logistic regression"])

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
selected,0.7970,+0.0060,0.0038,+1.6,6/8
selected + recency_ratio,0.7969,+0.0059,0.0038,+1.6,6/8
selected + boosting candidates,0.7969,+0.0059,0.0038,+1.6,5/8
all groups,0.7964,+0.0055,0.0044,+1.2,5/8
base,0.7910,+0.0000,-,-,ref


In [12]:
show(combine_tables["gradient boosting"])

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
selected,0.7859,+0.0131,0.0062,+2.1,6/8
selected + recency_ratio,0.7854,+0.0126,0.0038,+3.4,8/8
selected + boosting candidates,0.7833,+0.0106,0.0051,+2.1,7/8
all groups,0.7828,+0.0101,0.0070,+1.4,6/8
base,0.7728,+0.0000,-,-,ref


**Reading the boards**
- **Logistic regression:** nothing adds to the month. Every configuration that contains `calendar` scores the same (+0.0059 to +0.0060).
- **Gradient boosting:** adding `recency_ratio` to the month leaves the mean gain unchanged (+0.0126 vs +0.0131) but makes it **more consistent**: 8/8 folds and 3.4 SEM instead of 6/8 and 2.1. Adding the other two candidates, or every group, **lowers** the gain (+0.0106 and +0.0101). Extra columns that carry no new information add noise.
- **Decision: keep only `calendar`**, for both families. `recency_ratio` does not raise the mean score, and one shared feature set is simpler to maintain and explain. `recency_ratio` stays a candidate for the boosting models in `06_models`.

# <font color='#E8800A'>5. Effect on the official validation</font> <a class="anchor" id="official"></a>
[Back to TOC](#toc)

**Back to the official scheme** (3 yearly folds + 90-day gap) and to the Kaggle metric, as in the previous notebooks: base vs selected groups, with both models.

In [13]:
def official_summary_full(pipeline):
    """Like official_summary, on the full frame (identifiers are needed to build the features)."""
    scores, oof = evaluate(pipeline, official_folds, X_full, y)
    early = np.concatenate([va for _, va in official_folds[:-1]])
    last = official_folds[-1][1]
    t_early, _ = best_threshold(y.iloc[early], oof.iloc[early])
    return {"ROC AUC mean": scores["ROC AUC"].mean(),
            "ROC AUC fold 1 / 2 / 3": " / ".join(f"{v:.3f}" for v in scores["ROC AUC"]),
            "threshold (folds 1-2)": t_early,
            "honest F1 (fold 3)": f1_score(y.iloc[last], oof.iloc[last] >= t_early)}


official = pd.DataFrame({
    (model_name, version): official_summary_full(feature_pipeline(config, model_name))
    for model_name in RECIPES
    for version, config in [("base", {}), ("selected groups", {"groups": SELECTED})]
}).T
official.style.format({"ROC AUC mean": "{:.4f}", "threshold (folds 1-2)": "{:.2f}", "honest F1 (fold 3)": "{:.3f}"})

**Reading the results** (official scheme: 3 yearly folds, 90-day gap)

| Model | ROC AUC base → + month | Folds improved | Honest F1 base → + month |
|---|---|---|---|
| logistic regression | 0.783 → **0.785** (+0.002) | 2/3 | 0.781 → **0.790** |
| gradient boosting | 0.763 → **0.774** (+0.012) | 2/3 | 0.764 → **0.777** |

- **Gradient boosting gains the most** (+0.012 ROC AUC), as on the 8 folds. Trees use the 12 months to fit the cycle directly.
- **Fold 1 (2022) goes down for both models** (−0.011 logistic, −0.003 boosting), while folds 2–3 (2023, 2024–25) go up clearly. The cycle is weaker in some years, so the gain is real on average but not guaranteed every year. The folds closest to test (2–3) are the ones that improve.
- **The honest F1 on fold 3 rises for both models** (+0.009 and +0.013). That fold is the closest to the Kaggle test.

# <font color='#E8800A'>6. Summary of part A</font> <a class="anchor" id="summary"></a>
[Back to TOC](#toc)

### What feature engineering found

| Group | Logistic regression | Gradient boosting | Decision |
|---|---|---|---|
| `calendar` (month) | +0.0060, 6/8 | **+0.0131**, 2.1 SEM, 6/8 | **kept** |
| `cycle_position` (month mod 4) | +0.0064, 2.3 SEM | +0.0058 | alternative to the month, for linear models only |
| `recency_ratio` | +0.0001 | +0.0046 alone; with the month, steadier but not higher | candidate for boosting in `06` |
| `intensity`, `activity_trend` | ≈ 0 | +0.002 to +0.003 alone; lower the gain next to the month | not kept |
| `invoice`, `customer_since`, `customer_geo` | ≈ 0 | ≤ 0 | not kept |

**Main finding:** the purchases follow a **4-month cycle** (peaks in March, July, November with low repeat rates; high repeat rates two months before each peak), stable over 10 years. The month of the purchase is the only new feature that clearly helps. The other EDA-based ideas repeat information the model already has.

**Effect on the official validation:** logistic regression 0.783 → 0.785 ROC AUC (honest F1 0.790), gradient boosting 0.763 → 0.774 (honest F1 0.777).

### Lines for `docs/decisions.md`

| Decision | Alternatives considered | Evidence |
|---|---|---|
| Add the month of purchase (categorical, one-hot) | month mod 4; no calendar feature | `features_ludovico` §3: +0.006 logistic, +0.013 boosting (2.1 SEM); 4-month repeat cycle stable 2015–19 vs 2020–25; official folds 2/3 up, honest F1 +0.009 / +0.013 (§5) |
| Do not add recency ratio, activity trend, intensity, invoice ratios, `clientes.csv` attributes | each group alone and combined with the month | §3–4: ≈ 0 for logistic regression; for boosting they do not raise the gain next to the month |
| `clientes.csv` safe to use (no future information) | — | `customer_since_year` never later than the purchase or the first recorded purchase (§2) |

# Part B · Feature selection

# <font color='#E8800A'>7. Part B: the reference and the selectors</font> <a class="anchor" id="selection"></a>
[Back to TOC](#toc)

**Reference:** every feature + the month (the result of part A). Every selection method is compared with it, fold by fold, as in the week 4 practical.

**Where selection sits:** a `select` step **between preprocessing and the model**. It works on the encoded columns (each one-hot level is a column) and is **refitted inside every training fold**. Selection reads the target, so fitting it once on all the data would leak.

**Three families** (as in the practical), each with a few settings:
- **Filter:** looks at each column without a model. Variance, mutual information with the target, redundancy between columns.
- **Wrapper:** removes columns step by step, refitting a model each time (RFE).
- **Embedded:** the model selects while it fits. L1 logistic regression sets coefficients to exactly 0; a random forest ranks columns by importance.

Plus the **`random_noise` floor** (section 11) and the **vote** of the three families (section 12).

**The output is a strategy, not a list of columns:** the chosen selector is refitted on whatever data it receives.

In [14]:
from functools import partial

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (RFE, SelectFromModel, SelectKBest, SelectorMixin, VarianceThreshold,
                                       mutual_info_classif)
from sklearn.inspection import permutation_importance

BASE_CONFIG = {"groups": SELECTED}


def selection_pipeline(selector, model_name, config=BASE_CONFIG):
    """Feature pipeline of part A with a `select` step inserted just before the model."""
    steps = feature_pipeline(config, model_name).steps
    return Pipeline(steps[:-1] + [("select", selector), steps[-1]])


def width(fitted):
    """Number of columns the model receives in a fitted selection pipeline."""
    select = fitted.named_steps["select"]
    if select == "passthrough":
        return fitted.named_steps["prep"].transform(X_full.head(2).pipe(fitted.named_steps["features"].transform)).shape[1]
    return int(select.get_support().sum())


def compare_selectors(selectors, model_name, reference="all features", folds=None):
    """Like compare(), for selectors: paired table + mean number of columns kept."""
    folds = prep_folds if folds is None else folds
    scores, widths = {}, {}
    for name, selector in selectors.items():
        cv = cross_validate(selection_pipeline(selector, model_name), X_full, y, cv=folds,
                            scoring="roc_auc", n_jobs=-1, return_estimator=True)
        scores[name] = cv["test_score"]
        widths[name] = np.mean([width(est) for est in cv["estimator"]])
    scores = pd.DataFrame(scores, index=pd.RangeIndex(1, len(folds) + 1, name="fold"))

    stamp = datetime.now().isoformat(timespec="seconds")
    pd.DataFrame([{"time": stamp, "model": model_name, "experiment": f"select: {name}",
                   "config": json.dumps(BASE_CONFIG), "folds": len(folds), "roc_auc_mean": scores[name].mean(),
                   "roc_auc_folds": json.dumps(scores[name].round(5).tolist())} for name in scores]
                 ).to_csv(EXPERIMENT_LOG, mode="a", header=not EXPERIMENT_LOG.exists(), index=False)

    table = paired_table(scores, reference)
    table.insert(1, "columns kept", pd.Series(widths))
    return table, scores


def show_sel(table):
    return show(table).format({"columns kept": "{:.1f}"})


def scaled_l1(C):
    """L1 logistic regression on standardised columns (its coefficients are only comparable after scaling)."""
    return make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1, solver="saga", C=C, max_iter=5000))


L1_COEF = "named_steps.logisticregression.coef_"
SELECTION_RESULTS = {"logistic regression": {}, "gradient boosting": {}}  # name -> per-fold scores, for the final board

In [15]:
reference_width = {name: width(selection_pipeline("passthrough", name).fit(X_full, y)) for name in RECIPES}
print("columns the model receives with every feature + month:", reference_width)

columns the model receives with every feature + month: {'logistic regression': 84, 'gradient boosting': 84}


# <font color='#E8800A'>8. Filter methods</font> <a class="anchor" id="filter"></a>
[Back to TOC](#toc)

| Filter | Question it asks | Settings |
|---|---|---|
| `VarianceThreshold` | does the column vary at all? (measures spread, not relevance) | variance > 0.01 |
| `SelectKBest` + mutual information | how much does each column, alone, say about the target? | keep the best 20 / 35 / 50 |
| **correlation filter** | are two columns recording the same fact? Of each pair with \|Spearman ρ\| above the threshold, keep the one more related to the target | ρ > 0.9 / 0.8 |

A relevance ranking alone cannot see that two of its top columns are copies of each other. That is why the redundancy filter is a separate question.

In [16]:
class CorrelationFilter(BaseEstimator, SelectorMixin):
    """Redundancy filter: of every pair of columns with |Spearman rho| > threshold, drop the one less related to the target."""

    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y):
        ranks = pd.DataFrame(np.asarray(X, dtype=float)).rank().to_numpy()
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.nan_to_num(np.abs(np.corrcoef(ranks, rowvar=False)))
            relevance = np.nan_to_num(np.abs([np.corrcoef(col, y)[0, 1] for col in ranks.T]))
        keep = np.ones(ranks.shape[1], dtype=bool)
        for i in np.argsort(-relevance):          # most relevant first: it survives its pairs
            if keep[i]:
                partners = (corr[i] > self.threshold) & keep
                partners[i] = False
                keep[partners] = False
        self.support_ = keep
        return self

    def _get_support_mask(self):
        return self.support_


mi = partial(mutual_info_classif, random_state=RANDOM_STATE)
filters = {
    "all features": "passthrough",
    "variance > 0.01": VarianceThreshold(0.01),
    "MI best 20": SelectKBest(mi, k=20),
    "MI best 35": SelectKBest(mi, k=35),
    "MI best 50": SelectKBest(mi, k=50),
    "correlation < 0.9": CorrelationFilter(0.9),
    "correlation < 0.8": CorrelationFilter(0.8),
}
filter_tables = {}
for model_name in RECIPES:
    filter_tables[model_name], fold_scores = compare_selectors(filters, model_name)
    SELECTION_RESULTS[model_name] |= dict(fold_scores)
show_sel(filter_tables["logistic regression"])

,ROC AUC,columns kept,diff vs ref,paired SEM,diff / SEM,folds won
correlation < 0.8,0.797806,62.5,0.000845,0.001954,0.432586,3/8
correlation < 0.9,0.797362,69.0,0.000402,0.001834,0.218996,4/8
MI best 35,0.797202,35.0,0.000241,0.009203,0.026236,3/8
variance > 0.01,0.797166,71.6,0.000205,0.000350,0.587263,4/8
all features,0.796961,76.2,0.000000,nan,nan,ref
MI best 50,0.792459,50.0,-0.004502,0.002623,-1.716055,1/8
MI best 20,0.791398,20.0,-0.005563,0.009651,-0.576362,2/8


In [17]:
show_sel(filter_tables["gradient boosting"])

,ROC AUC,columns kept,diff vs ref,paired SEM,diff / SEM,folds won
variance > 0.01,0.788614,71.6,0.002738,0.003583,0.764022,4/8
all features,0.785877,76.2,0.000000,nan,nan,ref
correlation < 0.9,0.783485,69.0,-0.002392,0.004547,-0.526020,4/8
correlation < 0.8,0.781470,62.5,-0.004406,0.003648,-1.207981,2/8
MI best 35,0.774515,35.0,-0.011361,0.010095,-1.125386,4/8
MI best 50,0.772489,50.0,-0.013387,0.007246,-1.847531,3/8
MI best 20,0.761938,20.0,-0.023939,0.009907,-2.416472,2/8


**Reading the boards** (reference: all features, about 76 columns per fold)
- **Logistic regression: no filter changes the score.** Variance and the correlation filters remove 5–14 columns at no cost (+0.0002 to +0.0008, under 1 SEM). Mutual information with a fixed budget **loses** when it cuts hard (20 or 50 columns: −0.005), and is unstable (the SEM is 3–5 times larger than for the other filters).
- **Gradient boosting: every filter that removes informative columns loses**, down to −0.024 for the best 20 by mutual information. Only `variance > 0.01` is neutral (+0.0027 ± 0.0036), because it removes only near-empty one-hot columns.
- **Why mutual information fails:** it scores each column **alone**. Columns that are only useful in combination, such as the month levels or counts read together with the windows, get low scores and are dropped.

# <font color='#E8800A'>9. Wrapper method</font> <a class="anchor" id="wrapper"></a>
[Back to TOC](#toc)

**Recursive feature elimination (RFE):** fit a model, drop the 10% of columns with the smallest coefficients, refit, repeat until the chosen width. The ranking model is a standardised logistic regression for both instruments.

The width is a setting, so three are tried. As in the practical, **a width chosen on these folds carries no verdict**: picking the best of several widths on the same folds is optimistic. Section 12 checks the winner on the official folds.

In [18]:
def rfe(n):
    return RFE(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
               n_features_to_select=n, step=0.1, importance_getter=L1_COEF)


wrappers = {"all features": "passthrough", "RFE 20": rfe(20), "RFE 35": rfe(35), "RFE 50": rfe(50)}
wrapper_tables = {}
for model_name in RECIPES:
    wrapper_tables[model_name], fold_scores = compare_selectors(wrappers, model_name)
    SELECTION_RESULTS[model_name] |= dict(fold_scores)
show_sel(wrapper_tables["logistic regression"])

,ROC AUC,columns kept,diff vs ref,paired SEM,diff / SEM,folds won
all features,0.796961,76.2,0.000000,nan,nan,ref
RFE 35,0.796746,35.0,-0.000215,0.001734,-0.123966,4/8
RFE 50,0.796515,50.0,-0.000446,0.000685,-0.651237,2/8
RFE 20,0.793097,20.0,-0.003863,0.002763,-1.398162,2/8


In [19]:
show_sel(wrapper_tables["gradient boosting"])

,ROC AUC,columns kept,diff vs ref,paired SEM,diff / SEM,folds won
all features,0.785877,76.2,0.000000,nan,nan,ref
RFE 50,0.783418,50.0,-0.002459,0.006000,-0.409856,3/8
RFE 35,0.777299,35.0,-0.008577,0.004316,-1.987159,2/8
RFE 20,0.767613,20.0,-0.018264,0.005803,-3.147394,1/8


**Reading the boards**
- **Logistic regression:** RFE can go down to **35 columns at no cost** (−0.0002 ± 0.0017). At 20 columns it starts to lose (−0.004).
- **Gradient boosting: RFE loses at every width** (−0.002 at 50 columns, −0.018 at 20). Its ranking comes from a *linear* model, and the columns a linear model finds weak can still be useful to a tree through non-linear splits.

# <font color='#E8800A'>10. Embedded methods</font> <a class="anchor" id="embedded"></a>
[Back to TOC](#toc)

- **L1 logistic regression** (`SelectFromModel`): the L1 penalty sets the coefficients of unhelpful columns to exactly 0, and the columns with a non-zero coefficient are kept. A smaller `C` means a stronger penalty and fewer columns.
- **Random forest importance** (`SelectFromModel`): keep the columns whose impurity importance is above the median, or the mean, of all columns.

As the practical puts it, *a coefficient or an importance is a score, not a decision*: `SelectFromModel` turns it into a decision with a threshold.

In [20]:
embedded = {
    "all features": "passthrough",
    "L1 C=0.05": SelectFromModel(scaled_l1(0.05), importance_getter=L1_COEF),
    "L1 C=0.2": SelectFromModel(scaled_l1(0.2), importance_getter=L1_COEF),
    "L1 C=1": SelectFromModel(scaled_l1(1.0), importance_getter=L1_COEF),
    "forest > median": SelectFromModel(RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                              random_state=RANDOM_STATE), threshold="median"),
    "forest > mean": SelectFromModel(RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                            random_state=RANDOM_STATE), threshold="mean"),
}
embedded_tables = {}
for model_name in RECIPES:
    embedded_tables[model_name], fold_scores = compare_selectors(embedded, model_name)
    SELECTION_RESULTS[model_name] |= dict(fold_scores)
show_sel(embedded_tables["logistic regression"])

,ROC AUC,columns kept,diff vs ref,paired SEM,diff / SEM,folds won
L1 C=0.05,0.801807,21.8,0.004846,0.003659,1.324618,5/8
L1 C=0.2,0.801094,24.0,0.004133,0.003192,1.294713,5/8
forest > median,0.797849,38.1,0.000889,0.003039,0.292444,3/8
all features,0.796961,76.2,0.000000,nan,nan,ref
L1 C=1,0.795544,24.1,-0.001416,0.002878,-0.492096,4/8
forest > mean,0.789779,23.4,-0.007181,0.003069,-2.340341,2/8


In [21]:
show_sel(embedded_tables["gradient boosting"])

,ROC AUC,columns kept,diff vs ref,paired SEM,diff / SEM,folds won
all features,0.785877,76.2,0.000000,nan,nan,ref
L1 C=0.05,0.781593,19.4,-0.004284,0.004755,-0.900836,3/8
forest > median,0.775152,38.1,-0.010725,0.004442,-2.414348,1/8
L1 C=0.2,0.771895,20.8,-0.013981,0.005653,-2.473166,2/8
L1 C=1,0.767360,19.2,-0.018516,0.004171,-4.439173,0/8
forest > mean,0.762973,23.4,-0.022904,0.009516,-2.406880,2/8


**Reading the boards**
- **L1 is the only method that improves logistic regression:** `C=0.05` gives **+0.0048 ± 0.0037** (1.3 SEM, 5/8 folds) with **22 columns instead of 76**, and `C=0.2` gives +0.0041 with 24 columns. Removing the weak and redundant columns means fewer coefficients to estimate, so more stable ones. That is a regularisation effect.
- **Random forest importance:** neutral at the median cut (38 columns), loses at the mean cut (23 columns).
- **Gradient boosting: no embedded method helps.** Every L1 setting and both forest cuts lose (−0.004 to −0.023).

# <font color='#E8800A'>11. The random_noise floor</font> <a class="anchor" id="noise"></a>
[Back to TOC](#toc)

**Idea:** `random_noise` carries no information (EDA Step 20). If it is given to the model, any **source** column that the model finds **less useful than the noise** is useless too.

**How:** for every fold, fit the pipeline **with** `random_noise`, then measure the **permutation importance** of every source column on the validation rows: how much ROC AUC drops when that column is shuffled. Importance is averaged over the 8 folds, for both models.

In [22]:
def permutation_by_fold(model_name, n_repeats=5):
    """Mean permutation importance (ROC AUC drop) of every source column, random_noise included, over the folds."""
    config = {**BASE_CONFIG, "keep": ["random_noise"]}
    per_fold = []
    for tr, va in prep_folds:
        fitted = feature_pipeline(config, model_name).fit(X_full.iloc[tr], y.iloc[tr])
        X_val = X_full.iloc[va]
        result = permutation_importance(fitted, X_val, y.iloc[va], scoring="roc_auc", n_repeats=n_repeats,
                                        random_state=RANDOM_STATE, n_jobs=-1)
        # Shuffles the RAW columns: purchase_date stands for the month built from it; customer_id never reaches the model
        per_fold.append(pd.Series(result.importances_mean, index=X_val.columns).drop("customer_id"))
    return pd.concat(per_fold, axis=1).mean(axis=1).rename({"purchase_date": "purchase_date (-> month)"})


importance = pd.DataFrame({name: permutation_by_fold(name) for name in RECIPES})
noise_level = importance.loc["random_noise"]
below_noise = importance.le(noise_level, axis=1)
importance.assign(**{f"below noise ({name})": below_noise[name] for name in RECIPES}).sort_values(
    "logistic regression", ascending=False).style.format("{:+.4f}", subset=list(RECIPES))

,logistic regression,gradient boosting,below noise (logistic regression),below noise (gradient boosting)
purchase_count_365d,+0.0834,+0.0523,False,False
purchase_count_90d,+0.0550,+0.0040,False,False
prior_total_spend,+0.0307,+0.0084,False,False
total_tax,+0.0190,+0.0035,False,False
discount_line,+0.0121,+0.0084,False,False
purchase_date (-> month),+0.0120,+0.0096,False,False
transport_method,+0.0111,+0.0016,False,False
customer_tenure_days,+0.0088,-0.0008,False,True
total_gross,+0.0080,+0.0005,False,True
prior_purchase_count,+0.0078,+0.0089,False,False


In [23]:
# Source columns that BOTH models find no more useful than random_noise
useless = below_noise.all(axis=1)
USELESS = sorted(c for c in importance.index[useless] if c not in ("random_noise", "purchase_date (-> month)"))
print(f"noise level: {noise_level.round(4).to_dict()}")
print(f"{len(USELESS)} source columns at or below the noise for both models: {USELESS}")

noise_configs = {"all features": BASE_CONFIG, "drop columns below noise": {**BASE_CONFIG, "drop": USELESS}}
noise_tables = {model_name: compare(noise_configs, model_name, reference="all features") for model_name in RECIPES}
show(noise_tables["logistic regression"])

noise level: {'logistic regression': -0.0002, 'gradient boosting': 0.0006}
2 source columns at or below the noise for both models: ['n_lines', 'total_shipping']


,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
drop columns below noise,0.7975,+0.0005,0.0010,+0.5,4/8
all features,0.7970,+0.0000,-,-,ref


In [24]:
show(noise_tables["gradient boosting"])

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
drop columns below noise,0.7861,+0.0003,0.0023,+0.1,3/8
all features,0.7859,+0.0000,-,-,ref


**Reading the results**
- **The strongest columns are the same for both models:** `purchase_count_365d` far ahead, then recent activity (`purchase_count_90d`, `spend_365d`), history (`prior_total_spend`, `prior_purchase_count`) and the **month** (`purchase_date`), which is in the top 6 for both. That confirms part A.
- **Only `n_lines` and `total_shipping` are at or below `random_noise` for both models**, and dropping them changes nothing (+0.0005 and +0.0003). The models already give them almost no weight.
- **Each model also relies on columns the other ignores.** Logistic regression uses `total_tax`, `transport_method` and `customer_tenure_days`. Gradient boosting uses `prior_mean_gap_days`, `spend_90d` and `days_since_previous_purchase`, the gaps and recency that a linear model cannot exploit through thresholds. One column list for both families would hurt one of them.

# <font color='#E8800A'>12. Vote and final board</font> <a class="anchor" id="vote"></a>
[Back to TOC](#toc)

**Vote, 2 families out of 3** (as in the practical): one setting per family, each fitted on the same training fold. A column is kept if at least 2 of them keep it, so no single method decides alone.
- filter: correlation < 0.9
- wrapper: RFE 35
- embedded: L1 C=0.2

In [25]:
class VoteSelector(BaseEstimator, SelectorMixin):
    """Keep the columns selected by at least `min_votes` of the given selectors (all refitted on the same data)."""

    def __init__(self, selectors=(), min_votes=2):
        self.selectors = selectors
        self.min_votes = min_votes

    def fit(self, X, y):
        self.votes_ = np.sum([clone(selector).fit(X, y).get_support() for selector in self.selectors], axis=0)
        return self

    def _get_support_mask(self):
        return self.votes_ >= self.min_votes


vote = VoteSelector([CorrelationFilter(0.9), rfe(35), SelectFromModel(scaled_l1(0.2), importance_getter=L1_COEF)])
vote_tables = {}
for model_name in RECIPES:
    vote_tables[model_name], fold_scores = compare_selectors({"all features": "passthrough", "vote 2 of 3": vote},
                                                             model_name)
    SELECTION_RESULTS[model_name] |= dict(fold_scores)

**Final board:** every selector of sections 8–12 against `all features`, for both models.

In [26]:
def board(model_name):
    scores = pd.DataFrame(SELECTION_RESULTS[model_name])
    widths = pd.concat([t["columns kept"] for t in (filter_tables[model_name], wrapper_tables[model_name],
                                                     embedded_tables[model_name], vote_tables[model_name])])
    table = paired_table(scores, "all features")
    table.insert(1, "columns kept", widths[~widths.index.duplicated()])
    return table


show_sel(board("logistic regression"))

,ROC AUC,columns kept,diff vs ref,paired SEM,diff / SEM,folds won
L1 C=0.05,0.801807,21.8,0.004846,0.003659,1.324618,5/8
L1 C=0.2,0.801094,24.0,0.004133,0.003192,1.294713,5/8
vote 2 of 3,0.800213,35.0,0.003253,0.003200,1.016569,4/8
forest > median,0.797849,38.1,0.000889,0.003039,0.292444,3/8
correlation < 0.8,0.797806,62.5,0.000845,0.001954,0.432586,3/8
correlation < 0.9,0.797362,69.0,0.000402,0.001834,0.218996,4/8
MI best 35,0.797202,35.0,0.000241,0.009203,0.026236,3/8
variance > 0.01,0.797166,71.6,0.000205,0.000350,0.587263,4/8
all features,0.796961,76.2,0.000000,nan,nan,ref
RFE 35,0.796746,35.0,-0.000215,0.001734,-0.123966,4/8


In [27]:
show_sel(board("gradient boosting"))

,ROC AUC,columns kept,diff vs ref,paired SEM,diff / SEM,folds won
variance > 0.01,0.788614,71.6,0.002738,0.003583,0.764022,4/8
all features,0.785877,76.2,0.000000,nan,nan,ref
correlation < 0.9,0.783485,69.0,-0.002392,0.004547,-0.526020,4/8
RFE 50,0.783418,50.0,-0.002459,0.006000,-0.409856,3/8
L1 C=0.05,0.781593,19.4,-0.004284,0.004755,-0.900836,3/8
correlation < 0.8,0.781470,62.5,-0.004406,0.003648,-1.207981,2/8
vote 2 of 3,0.777370,32.5,-0.008506,0.004895,-1.737649,2/8
RFE 35,0.777299,35.0,-0.008577,0.004316,-1.987159,2/8
forest > median,0.775152,38.1,-0.010725,0.004442,-2.414348,1/8
MI best 35,0.774515,35.0,-0.011361,0.010095,-1.125386,4/8


**Reading the final boards**
- **Logistic regression:** the best selectors are **embedded L1** (`C=0.05`: +0.0048, 22 columns; `C=0.2`: +0.0041, 24 columns) and the **vote** (+0.0033, 35 columns). No selector reaches 2 SEM, but L1 and the vote are positive on 4–5 folds out of 8 with **a third to a half of the columns**.
- **Gradient boosting:** **nothing beats all features**. The more a method cuts, the more it loses.
- **The same lesson as the practical:** *selection does not improve prediction much, it changes the width of the model.* Here it helps a little for a linear model, which benefits from fewer and less redundant coefficients, and does not help a tree, which already chooses its columns at every split.

**Official check:** the best selector of each model on the official scheme (3 yearly folds, 90-day gap). Its setting was chosen on the 8 folds, so this is the test that it holds on a different split.

In [28]:
BEST_SELECTOR = {"logistic regression": "L1 C=0.05", "gradient boosting": "all features"}
ALL_SELECTORS = {**filters, **wrappers, **embedded, "vote 2 of 3": vote}

final_check = pd.DataFrame({
    (model_name, version): official_summary_full(selection_pipeline(clone(ALL_SELECTORS[name]) if name != "all features" else "passthrough", model_name))
    for model_name in RECIPES
    for version, name in [("all features", "all features"), (f"selected: {BEST_SELECTOR[model_name]}", BEST_SELECTOR[model_name])]
}).T
final_check.style.format({"ROC AUC mean": "{:.4f}", "threshold (folds 1-2)": "{:.2f}", "honest F1 (fold 3)": "{:.3f}"})

**Reading the results** (official scheme: 3 yearly folds, 90-day gap)

| Model | ROC AUC all features → selected | Per fold | Honest F1 |
|---|---|---|---|
| logistic regression + **L1 C = 0.05** | 0.785 → **0.796** (+0.011) | **3/3 up** (+0.012, +0.016, +0.004) | 0.790 → **0.802** |
| gradient boosting (no selection) | 0.774 | – | 0.777 |

- **On the official folds L1 selection gains more than on the 8 half-year folds** (+0.011 vs +0.005), and on every fold. The choice of `C = 0.05` was made on the 8 folds, so this is a check on a different split, not a re-use of the same one. It still covers the same years, so read it as a strong confirmation, not as an independent test.
- **Logistic regression with L1 selection is now the best pipeline so far** (honest F1 0.802), clearly above gradient boosting with default settings.

# <font color='#E8800A'>13. Summary of part B</font> <a class="anchor" id="summaryb"></a>
[Back to TOC](#toc)

### What feature selection found

| | Logistic regression | Gradient boosting |
|---|---|---|
| Best selector (8 folds) | **L1, C = 0.05**: +0.0048 ± 0.0037, **22 of 76 columns** | **none**: every method that removes columns loses |
| Filters | neutral (variance, correlation); mutual information loses when it cuts hard | lose, up to −0.024 |
| Wrapper (RFE) | neutral down to 35 columns | loses at every width |
| Embedded | **L1 improves**; forest importance neutral or loses | loses |
| Vote 2 of 3 | +0.0033, 35 columns | −0.0085 |
| Below `random_noise` | `n_lines`, `total_shipping` (dropping them: no change) | same |

### Decisions
- **Linear and distance-based models:** an L1 selection step (`SelectFromModel`, L1 logistic regression on standardised columns) between preprocessing and the model, refitted in every fold. Its strength (`C`) becomes a **hyperparameter to tune in `06_models`**: the gain is a regularisation effect, and it is cleaner to choose it together with the model.
- **Tree models:** **no selection**. Trees choose their split columns themselves, and every selector tested loses.
- **What is exported is the strategy, not a list of columns** (as in the practical): the selector is refitted on whatever rows it is trained on.

### About "a substantial improvement"
Feature selection gives nothing to gradient boosting, but **L1 selection is the largest single gain for logistic regression**: +0.005 on the 8 folds and **+0.011 on the official folds** (all 3 up). Here it works as a regularisation: 22 well-chosen columns give more stable coefficients than 76 partly redundant ones. In total, from the V01 baseline:

| Step | Logistic regression (ROC AUC, official folds) |
|---|---|
| V01 baseline | 0.778 |
| + preprocessing (`log1p`, robust, one-hot 50) | 0.783 |
| + month of purchase | 0.785 |
| + L1 selection (22 of 76 columns) | **0.796** (honest F1 0.802) |

The next substantial gains are more likely to come from **`06_models`**: algorithms that capture interactions and non-linearities (random forest, tuned boosting, SVM, neural networks), **tuning** their hyperparameters, and the decision **threshold**.

### Lines for `docs/decisions.md`

| Decision | Alternatives considered | Evidence |
|---|---|---|
| Linear models: L1 selection (`SelectFromModel`) inside the pipeline, strength tuned in `06` | variance, mutual information (k = 20/35/50), correlation filter (0.8/0.9), RFE (20/35/50), forest importance, vote 2 of 3 | `features_ludovico` §8–12: L1 C = 0.05 +0.0048 ± 0.0037 with 22 of 76 columns; official folds 0.785 → 0.796 ROC AUC (3/3 up), honest F1 0.790 → 0.802 |
| Tree models: no feature selection | same candidates | §8–12: every selector that removes columns loses (up to −0.024) |
| Keep `n_lines` and `total_shipping` (below `random_noise`) | drop them | §11: dropping them +0.0005 / +0.0003, no difference |

**The experiment log** (`data/processed/feature_experiments.csv`, not tracked by git) keeps every run of `compare`. To review it:

In [29]:
pd.read_csv(EXPERIMENT_LOG).tail(10)

,time,model,experiment,config,folds,roc_auc_mean,roc_auc_folds
106,2026-10-15T18:02:53,gradient boosting,select: variance > 0.01,"{""groups"": [""calendar""]}",8,0.788614,"[0.78076, 0.80499, 0.80462, 0.7608, 0.77747, 0..."
107,2026-10-15T18:02:53,gradient boosting,select: MI best 20,"{""groups"": [""calendar""]}",8,0.761938,"[0.7483, 0.77932, 0.80189, 0.76384, 0.7104, 0...."
108,2026-10-15T18:02:53,gradient boosting,select: MI best 35,"{""groups"": [""calendar""]}",8,0.774515,"[0.74619, 0.79415, 0.80706, 0.78051, 0.72476, ..."
109,2026-10-15T18:02:53,gradient boosting,select: MI best 50,"{""groups"": [""calendar""]}",8,0.772489,"[0.7517, 0.79309, 0.80044, 0.7518, 0.72599, 0...."
110,2026-10-15T18:02:53,gradient boosting,select: correlation < 0.9,"{""groups"": [""calendar""]}",8,0.783485,"[0.76801, 0.80823, 0.78148, 0.75802, 0.77041, ..."
111,2026-10-15T18:02:53,gradient boosting,select: correlation < 0.8,"{""groups"": [""calendar""]}",8,0.781470,"[0.78635, 0.80205, 0.78343, 0.74134, 0.74734, ..."
112,2026-10-15T18:02:57,logistic regression,select: all features,"{""groups"": [""calendar""]}",8,0.796961,"[0.79666, 0.83663, 0.79638, 0.74548, 0.79646, ..."
113,2026-10-15T18:02:57,logistic regression,select: RFE 20,"{""groups"": [""calendar""]}",8,0.793097,"[0.78788, 0.83324, 0.78643, 0.75752, 0.78391, ..."
114,2026-10-15T18:02:57,logistic regression,select: RFE 35,"{""groups"": [""calendar""]}",8,0.796746,"[0.79434, 0.83131, 0.79794, 0.74901, 0.79967, ..."
115,2026-10-15T18:02:57,logistic regression,select: RFE 50,"{""groups"": [""calendar""]}",8,0.796515,"[0.79939, 0.83381, 0.79522, 0.74522, 0.79362, ..."
